<a href="https://colab.research.google.com/github/PavanRishi69/dataviz-exercises-pavan-rishi-gillella/blob/main/lecture07_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [ ]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [5]:
# Task 1
import pandas as pd
import plotly.express as px

# 1. Prepare the data
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'
target_ratings = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(target_ratings)]

# 2. Pivot data to get counts by rating (y) and decade (x)
heatmap_data = (
    df_filtered.groupby(['rating', 'decade'])
    .size()
    .reset_index(name='count')
    .pivot(index='rating', columns='decade', values='count')
    .fillna(0)
)

# Reorder index/columns for better logical presentation if needed
heatmap_data = heatmap_data.reindex(index=target_ratings)

# 3. Create Heatmap
fig1 = px.imshow(
    heatmap_data,
    labels=dict(x="Release Decade", y="Content Rating", color="Title Count"),
    color_continuous_scale="Blues",
    text_auto=True,
    title="<b>The Rise of Mature Content: TV-MA Dominates the 2010s and 2020s Streaming Era</b>"
)

# 4. Customizing layout and adding an annotation
fig1.update_layout(
    title_font_size=16,
    xaxis_title="Release Decade",
    yaxis_title="Content Rating"
)

# Annotate the absolute highest peak cell directly
fig1.add_annotation(
    x="2010s",
    y="TV-MA",
    text="Peak Catalogue Density",
    showarrow=True,
    arrowhead=2,
    arrowcolor="black",
    ax=0,
    ay=-40,
    font=dict(color="black", size=11),
    bgcolor="white",
    bordercolor="black"
)

fig1.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [8]:
# Filter Movies added between 2015–2022
movies = df[
    (df['type'] == 'Movie') &
    (df['added_year'].between(2015, 2022))
]

# Count movies added each year
yearly_movies = (
    movies.groupby('added_year')
    .size()
    .reset_index(name='count')
    .sort_values('added_year')
)

# Find year with highest additions
max_idx = yearly_movies['count'].idxmax()
max_year = yearly_movies.loc[max_idx, 'added_year']
max_count = yearly_movies.loc[max_idx, 'count']

# Create waterfall chart
fig = go.Figure(go.Waterfall(
    x=[str(y) for y in yearly_movies['added_year']] + ['Total'],
    y=list(yearly_movies['count']) + [0],
    measure=['relative'] * len(yearly_movies) + ['total'],

    increasing={"marker": {"color": "green"}},
    decreasing={"marker": {"color": "red"}},
    totals={"marker": {"color": "blue"}}
))

# Add annotation for largest addition year
fig.add_annotation(
    x=str(max_year),
    y=max_count,
    text=f'Highest growth: {max_year}<br>{max_count} movies',
    showarrow=True,
    arrowhead=2
)

# Update layout
fig.update_layout(
    title='Netflix Movie Library Growth Accelerated Rapidly Before Reaching Maturity',
    xaxis_title='Year Added',
    yaxis_title='Movies Added',
    showlegend=False
)

fig.show()
